In [ ]:
# %%
# ================================
# AMP Regression Runner (TEA-seq)
# Target: CD45RA (continuous)
# Modalities:
#   HD: ATAC, RNA
#   LD: ADT-minus-target (features exclude the target protein)
# Label/target y:
#   from adt.h5ad (target protein column)
# Head:
#   linear regression (beta_hat = pinv(U_all) @ y_train) inside pipeline
# ================================

import sys
import json
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, Any, Tuple

import numpy as np
import pandas as pd
import scanpy as sc

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
)

In [ ]:
# ------------------------------------------------------------
# Project path + module import
# ------------------------------------------------------------
sys.path.append("../Python_scripts")

from multimodal_prediction_linear import MultimodalClusterAllUPipeline, predict_from_test_data_all

In [ ]:
# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
def evaluate_regression(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    with_rank_corr: bool = False
) -> Dict[str, float]:
    """
    Compute common regression metrics for test predictions.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    eps = 1e-12

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    evs  = float(explained_variance_score(y_true, y_pred))

    # Pearson
    yt = (y_true - y_true.mean()) / (y_true.std(ddof=0) + eps)
    yp = (y_pred - y_pred.mean()) / (y_pred.std(ddof=0) + eps)
    pearson_r = float(np.clip((yt * yp).mean(), -1.0, 1.0))

    out = {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "explained_var": evs,
        "pearson_r": pearson_r,
    }

    if with_rank_corr:
        try:
            from scipy.stats import spearmanr
            out["spearman_r"] = float(spearmanr(y_true, y_pred).correlation)
        except Exception:
            out["spearman_r"] = np.nan

    return out

In [ ]:
# ------------------------------------------------------------
# Split reader (same as your classification script)
# ------------------------------------------------------------
def read_index_csv(path: Path, n_total: Optional[int] = None) -> Dict[str, Any]:
    """
    Reads an index CSV containing a single numeric column.

    Returns:
      {
        "idx0": np.ndarray  # 0-based indices (keeps original order)
        "mode": str         # how indexing was interpreted
      }

    Policy:
      - If (min,max) == (0, n_total-1): treat as 0-based (sentinel match)
      - If (min,max) == (1, n_total):   treat as 1-based and convert
      - Otherwise: default to 0-based (because your files are generated 0-based)
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")

    # read w/ header, fallback no-header
    df = pd.read_csv(path)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        df = pd.read_csv(path, header=None)
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            raise ValueError(f"No numeric column found in {path}")

    idx = df[num_cols[0]].to_numpy()

    if idx.size == 0:
        raise ValueError(f"Empty index file: {path}")

    # require integer-valued (avoid silent truncation like 3.7 -> 3)
    if np.isnan(idx).any():
        bad = np.where(np.isnan(idx))[0][:10]
        raise ValueError(f"NaNs in {path} at rows {bad.tolist()}")

    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        bad = np.where(~np.isclose(idx, idx_int))[0][:10]
        raise ValueError(f"Non-integer indices in {path} at rows {bad.tolist()}: {idx[bad].tolist()}")

    idx = idx_int

    # decide base
    mode = "assumed_0-based"
    idx0 = idx

    if n_total is not None:
        mn, mx = int(idx.min()), int(idx.max())

        if mn == 0 and mx == (n_total - 1):
            mode = "0-based (sentinel match)"
            idx0 = idx
        elif mn == 1 and mx == n_total:
            mode = "1-based->converted_to_0-based (sentinel match)"
            idx0 = idx - 1
        else:
            # ambiguous partial range => default to 0-based
            mode = f"assumed_0-based (mn={mn}, mx={mx})"
            idx0 = idx

        # range check (NO clipping)
        if (idx0 < 0).any() or (idx0 >= n_total).any():
            bad = np.where((idx0 < 0) | (idx0 >= n_total))[0][:10]
            raise ValueError(
                f"Out-of-range indices in {path} after base handling. "
                f"mode={mode}, n_total={n_total}, min={idx0.min()}, max={idx0.max()}, "
                f"examples rows {bad.tolist()} values {idx0[bad].tolist()}."
            )

    return {"idx0": idx0.astype(np.int64), "mode": mode}

In [ ]:
# ------------------------------------------------------------
# Data loader for regression with ADT-minus-target features
# ------------------------------------------------------------
def load_tea_norm_matrices_for_regression(base_path: Path, target_protein: str):
    """
    Loads:
      rna.h5ad
      atac.h5ad
      adt_minus_<target>.h5ad (LD features)
      response/<target>.csv   (y target)

    Returns:
      X_rna, X_atac, X_adt_minus as float64 arrays
      y_target as float32 vector (aligned to cell_ids)
      cell_ids
    """
    base_path = Path(base_path)

    rna  = sc.read_h5ad(base_path / "rna.h5ad")
    atac = sc.read_h5ad(base_path / "atac.h5ad")
    adt_minus = sc.read_h5ad(base_path / f"adt_minus_{target_protein}.h5ad")

    # ---- obs alignment ----
    if not np.array_equal(rna.obs_names.values, atac.obs_names.values):
        raise RuntimeError("rna and atac obs_names mismatch")
    if not np.array_equal(rna.obs_names.values, adt_minus.obs_names.values):
        raise RuntimeError("rna and adt_minus obs_names mismatch")

    # ---- norm layer ----
    for nm, obj in [("rna", rna), ("atac", atac), ("adt_minus", adt_minus)]:
        if "norm" not in obj.layers:
            raise KeyError(f"Expected .layers['norm'] in {nm}.h5ad")

    def _to_dense(X):
        return X.toarray() if hasattr(X, "toarray") else np.asarray(X)

    X_rna  = _to_dense(rna.layers["norm"]).astype(np.float64, copy=False)
    X_atac = _to_dense(atac.layers["norm"]).astype(np.float64, copy=False)
    X_adt_minus = _to_dense(adt_minus.layers["norm"]).astype(np.float64, copy=False)

    cell_ids = rna.obs_names.to_numpy()
    cell_names = rna.obs_names.astype(str).to_list()  # for pandas indexing

    # ---- target y from response CSV (correct) ----
    y_path = base_path / "response" / f"{target_protein}.csv"
    df_y = pd.read_csv(y_path, index_col=0)

    # align by cell IDs; fails loudly if mismatch
    try:
        y = df_y.loc[cell_names].iloc[:, 0].values.astype(np.float32)
    except KeyError as e:
        missing = sorted(set(cell_names) - set(df_y.index.astype(str)))
        raise KeyError(
            f"response CSV index does not cover all cells in AnnData. "
            f"Missing {len(missing)} cell IDs (showing first 10): {missing[:10]}"
        ) from e

    return X_rna, X_atac, X_adt_minus, y, cell_ids

In [ ]:
# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------
def diag_nan_inf(name, X):
    X = np.asarray(X)
    n_nan = int(np.isnan(X).sum())
    n_inf = int(np.isinf(X).sum())
    mn = float(np.nanmin(X))
    mx = float(np.nanmax(X))
    print(f"[Diag] {name}: nan={n_nan} inf={n_inf} min={mn:.3g} max={mx:.3g}")

In [ ]:
# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
dataset_name = "tea"
split_tag    = "tea_split3_all_celltypes"

TARGET_PROTEIN = "CD45RA"

base_dir    = Path("../data")     
splits_dir  = Path("../splits")
results_dir = Path("../results")  / dataset_name

SIMILARITY     = "cka"
NUM_CLUSTERS   = 1
AMP_ITERS      = 10
RANDOM_STATE   = 12

RELATION = "linear"  # must be "linear" to use beta_hat regression head

results_root = results_dir / f"{split_tag}_amp_early_fusion_reg_{TARGET_PROTEIN}_{RELATION}"
results_root.mkdir(parents=True, exist_ok=True)
print("Results root:", results_root)

In [ ]:
# %%
# ------------------------------------------------------------
# Load full matrices + target
# ------------------------------------------------------------
X_rna_full, X_atac_full, X_adtm_full, y_full, cell_ids = load_tea_norm_matrices_for_regression(
    base_dir, target_protein=TARGET_PROTEIN
)

n_cells = X_rna_full.shape[0]
print("[Data] n_cells:", n_cells)
print("[Data] RNA :", X_rna_full.shape)
print("[Data] ATAC:", X_atac_full.shape)
print("[Data] ADT_minus:", X_adtm_full.shape)
print("[Data] y target:", y_full.shape, "min/max:", float(np.min(y_full)), float(np.max(y_full)))

# basic diagnostics
diag_nan_inf("RNA_full", X_rna_full)
diag_nan_inf("ATAC_full", X_atac_full)
diag_nan_inf("ADT_minus_full", X_adtm_full)
diag_nan_inf("y_full", y_full)

In [ ]:
# %%
# ------------------------------------------------------------
# Read splits
# ------------------------------------------------------------
train_path = splits_dir / f"{split_tag}_train_idx.csv"
test_path  = splits_dir / f"{split_tag}_test_idx.csv"

if not train_path.exists():
    raise FileNotFoundError(f"Missing: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Missing: {test_path}")

tr_res = read_index_csv(train_path, n_total=n_cells)
te_res = read_index_csv(test_path,  n_total=n_cells)

idx_tr0 = tr_res["idx0"]
idx_te0 = te_res["idx0"]

print("[Splits] TRAIN idx mode:", tr_res["mode"], "| n_train:", len(idx_tr0))
print("[Splits] TEST  idx mode:", te_res["mode"], "| n_test :", len(idx_te0))

# sanity
if np.any(idx_tr0 < 0) or np.any(idx_tr0 >= n_cells):
    raise ValueError("Train indices out of range.")
if np.any(idx_te0 < 0) or np.any(idx_te0 >= n_cells):
    raise ValueError("Test indices out of range.")
overlap = np.intersect1d(idx_tr0, idx_te0)
print("[Splits] overlap(train,test):", overlap.size)

In [ ]:
# %%
# ------------------------------------------------------------
# Slice train/test
# ------------------------------------------------------------
y_train = y_full[idx_tr0]
y_test  = y_full[idx_te0]

X_rna_tr  = X_rna_full[idx_tr0, :]
X_atac_tr = X_atac_full[idx_tr0, :]
X_adtm_tr = X_adtm_full[idx_tr0, :]

X_rna_te  = X_rna_full[idx_te0, :]
X_atac_te = X_atac_full[idx_te0, :]
X_adtm_te = X_adtm_full[idx_te0, :]

print("[Train] RNA/ATAC/ADT_minus:", X_rna_tr.shape, X_atac_tr.shape, X_adtm_tr.shape)
print("[Test ] RNA/ATAC/ADT_minus:", X_rna_te.shape, X_atac_te.shape, X_adtm_te.shape)

diag_nan_inf("RNA_train", X_rna_tr)
diag_nan_inf("ATAC_train", X_atac_tr)
diag_nan_inf("ADT_minus_train", X_adtm_tr)
diag_nan_inf("y_train", y_train)

In [ ]:
y_test.shape

In [ ]:
# %%
# ------------------------------------------------------------
# Arrange modalities for pipeline
#   HD: ATAC + RNA
#   LD: ADT_minus_target
# ------------------------------------------------------------
B_hd_train = [X_atac_tr, X_rna_tr]
B_hd_test  = [X_atac_te, X_rna_te]
A_ld_train = [X_adtm_tr]
A_ld_test  = [X_adtm_te]

# ranks (keep your existing choices)
K_ld = 10
K_B  = 15
K_C  = 20
K_hd_list = [K_B, K_C]
K_ld_list = [K_ld]

print("[Config] K_hd_list:", K_hd_list, "K_ld_list:", K_ld_list)


In [ ]:
# %%
# ------------------------------------------------------------
# Fit AMP pipeline on TRAIN (regression, linear head)
# ------------------------------------------------------------
pipe = MultimodalClusterAllUPipeline()
pipe.task = "regression"
pipe.relation = "linear"     # IMPORTANT: triggers beta_hat in _post_amp_supervised
pipe.y_train = y_train       # must be set before run_amp

print("[AMP] Fitting PCA (HD) on TRAIN only...")
pipe.fit_pca_highdim(B_hd_train, K_hd_list, preprocess=False)

print("[AMP] Fitting LD loadings on TRAIN only...")
pipe.fit_lowdim(A_ld_train)

print("[AMP] Clustering TRAIN modalities...")
labels_all = pipe.cluster_all_modalities(
    X_list_hd=B_hd_train,
    similarity_metric=SIMILARITY,
    num_clusters=NUM_CLUSTERS,
    threshold=None
)
print("[AMP] cluster_labels_U_all:", labels_all)

print("[AMP] Building EB models on TRAIN...")
pipe.build_cluster_models(B_hd_train, print_priors=False)

print("[AMP] Running AMP on TRAIN...")
amp_res = pipe.run_amp(
    X_list_hd=B_hd_train,
    amp_iters=AMP_ITERS,
    muteu=False,
    mutev=False
)
print("[AMP] AMP keys:", sorted(amp_res.keys()))
if "beta_hat" not in amp_res:
    raise RuntimeError("Expected beta_hat in amp_results for linear regression head. Check pipe.relation == 'linear'.")

In [ ]:
# %%
# ------------------------------------------------------------
# Predict on TEST
# ------------------------------------------------------------
print(f"[Predict] Predicting TEST target: {TARGET_PROTEIN} ...")
U_hd_den_te, U_ld_den_te, y_pred = predict_from_test_data_all(
    pipeline=pipe,
    X_test_hd=B_hd_test,
    X_test_ld=A_ld_test
)

y_pred = np.asarray(y_pred).ravel()
y_true = np.asarray(y_test).ravel()

print("[Predict] y_pred:", y_pred.shape, "min/max:", float(np.min(y_pred)), float(np.max(y_pred)))
print("[Predict] y_true:", y_true.shape, "min/max:", float(np.min(y_true)), float(np.max(y_true)))


In [ ]:
# %%
# ------------------------------------------------------------
# Evaluate + save
# ------------------------------------------------------------
report = evaluate_regression(y_true, y_pred, with_rank_corr=True)
print("[Metrics]", report)

stamp = datetime.now().isoformat(timespec="seconds").replace(":", "-")

# Save arrays
np.save(results_root / "amp_y_hat.npy", y_pred)
np.save(results_root / "amp_y_true.npy", y_true)
np.save(results_root / "amp_idx_test.npy", idx_te0.astype(int))

# Save CSV predictions
test_cell_names = cell_ids[idx_te0]
pred_df = pd.DataFrame({
    "cell": test_cell_names,
    "y_true": y_true,
    "y_pred": y_pred,
})
pred_df.to_csv(results_root / "amp_test_predictions.csv", index=False)

# Save metrics JSON
def _to_serializable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating, np.integer)):
        return float(x)
    return x

report_json = {k: _to_serializable(v) for k, v in report.items()}
report_json["run"] = {
    "pipeline": "AMP-HD+LD",
    "method": "early_fusion",
    "task": "regression",
    "relation": RELATION,
    "target": TARGET_PROTEIN,
    "dataset": dataset_name,
    "split_tag": split_tag,
    "similarity": SIMILARITY,
    "num_clusters": NUM_CLUSTERS,
    "amp_iters": AMP_ITERS,
    "K_hd_list": K_hd_list,
    "K_ld_list": K_ld_list,
}
report_json["timestamp"] = datetime.now().isoformat(timespec="seconds")
report_json["split_idx_mode"] = {"train": tr_res["mode"], "test": te_res["mode"]}

with open(results_root / "amp_test_metrics.json", "w") as f:
    json.dump(report_json, f, indent=2)

print("[Saved] outputs under:", results_root)
print("  - amp_y_hat.npy / amp_y_true.npy / amp_idx_test.npy")
print("  - amp_test_predictions.csv")
print("  - amp_test_metrics.json")
# %%


In [ ]:
# ------------------------------------------------------------
# Per-cell-type metrics
# ------------------------------------------------------------
meta = pd.read_csv("../data/cleaned_cell_labels_meta_tea_seq.csv", index_col=0)
meta.columns = ["CellType"]
celltype_map = meta["CellType"].to_dict()

pred_df["CellType"] = pred_df["cell"].map(celltype_map)
pred_df_ct = pred_df

def evaluate_regression_df(df):
    yt = df["y_true"].to_numpy()
    yp = df["y_pred"].to_numpy()
    from scipy.stats import spearmanr
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mae  = float(mean_absolute_error(yt, yp))
    r2   = float(r2_score(yt, yp))
    pr   = float(np.corrcoef(yt, yp)[0, 1])
    sr   = float(spearmanr(yt, yp)[0])
    return {"n": len(df), "rmse": round(rmse, 4), "mae": round(mae, 4),
            "r2": round(r2, 4), "pearson_r": round(pr, 4), "spearman_r": round(sr, 4)}

rows = []

# overall
m = evaluate_regression_df(pred_df_ct)
rows.append({"cell_type": "OVERALL", **m})

# per cell type
for ct, grp in pred_df_ct.groupby("CellType"):
    if len(grp) < 5:
        continue
    m = evaluate_regression_df(grp)
    rows.append({"cell_type": ct, **m})

ct_metrics = pd.DataFrame(rows).set_index("cell_type")
print(ct_metrics.to_string())
ct_metrics.to_csv(results_root / "amp_test_metrics_by_celltype.csv")

In [ ]:
# ------------------------------------------------------------
# UMAP of denoised test features
# ------------------------------------------------------------
import umap as umap_lib

# Stack denoised U: HD modalities (sorted keys) + LD modalities (sorted keys)
U_hd_parts = [U_hd_den_te[k] for k in sorted(U_hd_den_te)]
U_ld_parts  = [U_ld_den_te[j] for j in sorted(U_ld_den_te)]
U_stack = np.hstack(U_hd_parts + U_ld_parts)
print("[UMAP] Input matrix shape:", U_stack.shape)

reducer = umap_lib.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.3,
    metric="cosine", random_state=42
)
emb = reducer.fit_transform(U_stack)

umap_df = pd.DataFrame({
    "cell":      test_cell_names,
    "umap_1":    emb[:, 0],
    "umap_2":    emb[:, 1],
    "y_true":    y_true,
    "y_pred":    y_pred,
    "abs_error": np.abs(y_true - y_pred),
})
umap_df["CellType"] = umap_df["cell"].map(celltype_map)

umap_csv = results_root / "amp_test_umap_embedding.csv"
umap_df.to_csv(umap_csv, index=False)
print("[Saved] UMAP embedding →", umap_csv)
print(umap_df.head())